---

### NOTEBOOK 01 : ANALYSE EXPLORATOIRE ET PRÉPARATION DES DONNÉES
 

    PROJET      : Détection d'anomalies dans les services financiers numériques en Haïti
    GROUPE      : G3
    MEMBRES     : Blemy JOSEPH, Jonas CLOCIN, Martin FRANÇOIS
    ENCADRANT   : Evens TOUSSAINT
    DATE        : 25/06/2026
    VERSION     : 1.0

---

---

## Description

    Ce notebook constitue le livrable central de la phase d'ingénierie des données.
    Il couvre les étapes 2 et 3 du projet:

    ÉTAPE 2 - PRÉPARATION DES DONNÉES:
        - Compréhension et évaluation des données
        - Documentation des variables (dictionnaire des données)
        - Détection des valeurs manquantes, doublons, erreurs de format
        - Définition et justification des règles de nettoyage
        - Exécution du nettoyage avec traçabilité

    ÉTAPE 3 - ANALYSE EXPLORATOIRE:
        - Statistiques descriptives
        - Étude des distributions
        - Analyse des relations entre variables
        - Détection préliminaire des anomalies
        - Synthèse et recommandations

    OBJECTIFS:
        - Assurer la transparence des choix de modélisation
        - Garantir la traçabilité des opérations de nettoyage
        - Documenter la structure des données pour les modèles non supervisés

---

### SECTION 1 : INTRODUCTION - STRATÉGIE D'ÉMULATION DES DONNÉES

### CONTEXTE ET PROBLÉMATIQUE

L'acquisition de données réelles du secteur bancaire haïtien s'est heurtée à des contraintes majeures :

1. **Secret bancaire et réglementations BRH**  
   Les institutions financières en Haïti sont soumises à des réglementations rigoureuses imposées par la Banque de la République d'Haïti (BRH) concernant la confidentialité des données.

2. **Silence de l'administration**  
   Une demande officielle d'accompagnement a été transmise à l'administration sans réponse à ce jour, confirmant l'absence de protocoles pour le partage de données à des fins de recherche.

3. **Inexistence de dépôts de données publics**  
   Aucun historique de transactions publiques n'existe pour le contexte économique haïtien.

---

#### Stratégie d'émulation (Dataset Proxy)

Face à ces obstacles, nous adoptons une approche par **Dataset Proxy** :

- Utilisation du dataset *"Indian Banking"* publiquement disponible
- Validation de la logique algorithmique 
- Émulation structurelle des opérations bancaires
- Pipeline ETL transposable aux données locales

---

#### Justification du choix - Indian Banking Dataset

| Critère | Évaluation |
|---------|------------|
| Diversité opérationnelle | Dépôts, retraits, virements |
| Multi-canaux | Guichets, ATM, web, mobile |
| Volume significatif | 550 000 transactions |
| Variables essentielles | Présentes |
| Absence de devise HTG | Ajustement des seuils possible |

---

#### Source du dataset

- **Nom** : Indian Banking Transactions Dataset  
- **Source** : Publiquement disponible sur [Kaggle](https://www.kaggle.com/datasets/belbino/indian-banking-transactions-20192024/)  
- **Type** : Données synthétiques simulant des transactions bancaires réelles

### SECTION 2 : IMPORTATION DES BIBLIOTHÈQUES

In [ ]:
"""

Cette section importe toutes les bibliothèques nécessaires pour:
    - La manipulation des données (pandas, numpy)
    - La visualisation (matplotlib, seaborn)
    - La gestion des fichiers et dossiers (pathlib)
    - La gestion des avertissements (warnings)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
from pathlib import Path

# Configuration des warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12


print("="*60)
print("NOTEBOOK 01 : ANALYSE EXPLORATOIRE ET PRÉPARATION DES DONNÉES")
print("="*60)
print(f"\nDate d'exécution : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nBibliothèques importées avec succès\n")
print(f"   - Pandas : {pd.__version__}")
print(f"   - NumPy : {np.__version__}")
print(f"   - Matplotlib : {plt.matplotlib.__version__}")
print(f"   - Seaborn : {sns.__version__}")

NOTEBOOK 01: ANALYSE EXPLORATOIRE ET PRÉPARATION DES DONNÉES

Date d'exécution : 2026-06-25 01:59:23

Bibliothèques importées avec succès

   - Pandas : 2.3.3
   - NumPy : 2.0.1
   - Matplotlib : 3.9.4
   - Seaborn : 0.13.2


### SECTION 3 : CHARGEMENT DES DONNÉES

In [34]:
"""
    Cette section charge le jeu de données à partir du fichier CSV.
    Le fichier source se trouve dans le dossier data/raw/.
"""

def load_data(filepath: str) -> pd.DataFrame:
    """
    Charge le jeu de données à partir d'un fichier CSV.
    
    Parameters
    ----------
    filepath : str
        Chemin absolu ou relatif vers le fichier CSV
        
    Returns
    -------
    pd.DataFrame
        DataFrame contenant les données brutes chargées depuis le fichier
        
    Raises
    ------
    FileNotFoundError
        Si le fichier spécifié n'existe pas
    Exception
        Pour toute autre erreur lors du chargement
        
    Examples
    --------
    >>> df = load_data('../data/raw/indian_banking_transactions.csv')
    >>> print(df.shape)
    (550000, 20)
    
    Notes
    -----
    Cette fonction utilise pandas.read_csv() pour charger les données.
    Les paramètres par défaut sont utilisés pour la lecture.
    """
    try:
        df = pd.read_csv(filepath)
        print(f"Données chargées avec succès\n")
        print(f"   - Fichier: {filepath}")
        print(f"   - Dimensions: {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
        return df
    except FileNotFoundError:
        print(f"Erreur: Fichier non trouvé à {filepath}")
        raise
    except Exception as e:
        print(f"Erreur lors du chargement: {str(e)}")
        raise

# Chargement des données
DATA_PATH = '../data/raw/indian_banking_transactions.csv'

try:
    df_raw = load_data(DATA_PATH)
    df = df_raw.copy()
    print("\nAperçu des 5 premières lignes:")
    display(df.head())
except Exception as e:
    print(f"Erreur de chargement: {e}")
    print("Création d'un DataFrame vide pour continuer...")
    df = pd.DataFrame()

Données chargées avec succès

   - Fichier: ../data/raw/indian_banking_transactions.csv
   - Dimensions: 550,000 lignes x 20 colonnes

Aperçu des 5 premières lignes:


,transaction_id,customer_id,transaction_date,transaction_time,account_type,transaction_type,transaction_amount,transaction_direction,account_balance,merchant_category,state,credit_score,has_loan,loan_type,emi_amount,transaction_status,channel,kyc_status,is_fraud,transaction_hour
0,TXN000000001,CUST015796,2019-01-01,15:28,Current,UPI,1820.17,Debit,609365.31,Food & Dining,Maharashtra,764,0,NaN,0.00,Success,Branch,Verified,0,15
1,TXN000000002,CUST000861,2019-01-01,03:00,Current,UPI,392.67,Credit,14451.14,Education,West Bengal,630,0,NaN,0.00,Success,Mobile_App,Verified,0,3
2,TXN000000003,CUST076821,2019-01-01,18:03,Fixed Deposit,POS,1255.97,Debit,47621.87,Utilities,Punjab,813,1,Home,1433.91,Success,Mobile_App,Verified,0,18
3,TXN000000004,CUST054887,2019-01-01,08:03,Savings,UPI,2580.68,Debit,34467.85,Travel,Karnataka,628,1,Personal,6280.35,Success,API,Verified,0,8
4,TXN000000005,CUST006266,2019-01-01,14:23,Fixed Deposit,UPI,2573.80,Debit,26617.40,Utilities,West Bengal,767,0,NaN,0.00,Success,API,Verified,0,14


### SECTION 4 : ANALYSE DE LA STRUCTURE DES DONNEES

In [35]:
"""
    Cette section examine la structure du jeu de donnees:
        - Informations generales (lignes, colonnes, memoire)
        - Types de donnees de chaque colonne
        - Liste complete des colonnes
        - Statistiques rapides pour les variables numeriques
"""

def analyze_structure(df: pd.DataFrame) -> dict:
    """
    Analyse complete de la structure du DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame a analyser
        
    Returns
    -------
    dict
        Dictionnaire contenant les informations de structure:
        - n_rows: Nombre de lignes
        - n_cols: Nombre de colonnes
        - dtypes: Types de donnees par colonne
        - columns: Liste des noms de colonnes
        - numeric_cols: Liste des colonnes numeriques
        
    Examples
    --------
    >>> info = analyze_structure(df)
    >>> print(info['n_rows'])
    550000
    
    Notes
    -----
    Cette fonction affiche egalement les resultats dans la console.
    """
    print("\n" + "="*60)
    print("4. ANALYSE DE LA STRUCTURE DES DONNEES")
    print("="*60)
    
    if df.empty:
        print("DataFrame vide")
        return {}
    
    # Informations generales
    print("\n4.1 INFORMATIONS GENERALES :")
    print("-"*40)
    print(f"   - Nombre de lignes: {df.shape[0]:,}")
    print(f"   - Nombre de colonnes: {df.shape[1]}")
    print(f"   - Memoire utilisee: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Types de donnees
    print("\n4.2 TYPES DE DONNEES :")
    print("-"*40)
    dtype_counts = df.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"   - {dtype} : {count} colonnes")
    
    # Liste des colonnes
    print("\n4.3 LISTE DES COLONNES :")
    print("-"*40)
    for i, col in enumerate(df.columns, 1):
        print(f"   {i:2d}. {col} ({df[col].dtype})")
    
    # Statistiques rapides
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print("\n4.4 STATISTIQUES RAPIDES (colonnes numeriques):")
        print("-"*40)
        display(df[numeric_cols].describe().round(2))
    
    return {
        'n_rows' : df.shape[0],
        'n_cols' : df.shape[1],
        'dtypes' : dtype_counts.to_dict(),
        'columns' : df.columns.tolist(),
        'numeric_cols' : numeric_cols.tolist()
    }

structure_info = analyze_structure(df)



4. ANALYSE DE LA STRUCTURE DES DONNEES

4.1 INFORMATIONS GENERALES :
----------------------------------------
   - Nombre de lignes: 550,000
   - Nombre de colonnes: 20
   - Memoire utilisee: 459.62 MB

4.2 TYPES DE DONNEES :
----------------------------------------
   - object : 13 colonnes
   - int64 : 4 colonnes
   - float64 : 3 colonnes

4.3 LISTE DES COLONNES :
----------------------------------------
    1. transaction_id (object)
    2. customer_id (object)
    3. transaction_date (object)
    4. transaction_time (object)
    5. account_type (object)
    6. transaction_type (object)
    7. transaction_amount (float64)
    8. transaction_direction (object)
    9. account_balance (float64)
   10. merchant_category (object)
   11. state (object)
   12. credit_score (int64)
   13. has_loan (int64)
   14. loan_type (object)
   15. emi_amount (float64)
   16. transaction_status (object)
   17. channel (object)
   18. kyc_status (object)
   19. is_fraud (int64)
   20. transaction_hour

,transaction_amount,account_balance,credit_score,has_loan,emi_amount,is_fraud,transaction_hour
count,550000.00,550000.00,550000.00,550000.00,550000.00,550000.00,550000.00
mean,29907.09,84550.12,599.81,0.35,2365.86,0.01,11.50
std,139610.10,170914.59,173.05,0.48,4973.55,0.09,6.92
min,2.40,500.00,300.00,0.00,0.00,0.00,0.00
25%,763.04,15157.78,450.00,0.00,0.00,0.00,6.00
50%,2035.74,36422.87,600.00,0.00,0.00,0.00,11.00
75%,8557.50,87607.04,750.00,1.00,3115.03,0.00,17.00
max,10000000.00,5000000.00,899.00,1.00,153325.74,1.00,23.00


### SECTION 5 : DICTIONNAIRE DES VARIABLES

In [36]:
"""
    Cette section documente chaque variable du jeu de donnees:
        - Nom de la variable
        - Type de donnee (numerique, categoriel, temporel, booleen)
        - Description detaillee
        - Exemple de valeur
        - Statistiques de base (valeurs manquantes, valeurs uniques, min, max)
"""

def create_data_dictionary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cree un dictionnaire des donnees avec descriptions detaillees.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame source
        
    Returns
    -------
    pd.DataFrame
        DataFrame contenant le dictionnaire des donnees avec:
        - Variable: Nom de la colonne
        - Type_Technique: Type Python de la colonne
        - Type_Donnee: Type conceptuel (numerique, categoriel, etc.)
        - Description: Description detaillee de la variable
        - Valeurs_manquantes: Nombre de valeurs manquantes
        - Taux_manquants: Pourcentage de valeurs manquantes
        - Valeurs_uniques: Nombre de valeurs uniques
        - Min: Valeur minimale
        - Max: Valeur maximale
        
    Examples
    --------
    >>> df_dict = create_data_dictionary(df)
    >>> df_dict.head()
    
    Notes
    -----
    Les descriptions sont basees sur la documentation du projet.
    """
    print("\n" + "="*60)
    print("5. DICTIONNAIRE DES VARIABLES")
    print("="*60)
    
    if df.empty:
        print("DataFrame vide")
        return pd.DataFrame()
    
    # Definitions des variables
    variable_descriptions = {
        'transaction_id': {
            'type': 'Categoriel', 
            'description': 'Identifiant unique attribue a chaque transaction financiere. Variable technique indispensable pour assurer l\'auditabilite et le tracage de chaque alerte generee.'
        },
        'customer_id': {
            'type': 'Categoriel', 
            'description': 'Identifiant unique du client. Variable pivot qui permet d\'agreger les transactions par individu afin de profiler son comportement historique.'
        },
        'transaction_date': {
            'type': 'Temporel', 
            'description': 'Date calendaire a laquelle l\'operation a ete enregistree. Permet d\'analyser les comportements selon le jour de la semaine ou les periodes du mois (ex: jours de paie).'
        },
        'transaction_time': {
            'type': 'Temporel', 
            'description': 'Heure precise (minutes, secondes) de l\'execution de la transaction, essentielle pour situer l\'action dans la journee.'
        },
        'account_type': {
            'type': 'Categoriel', 
            'description': 'Type de compte bancaire sur lequel l\'operation est enregistree, ce qui permet de contextualiser les plafonds autorises. Types: Savings, CURRENT, Salary, NRI, Fixed Deposit.'
        },
        'transaction_type': {
            'type': 'Categoriel', 
            'description': 'Nature commerciale ou technique de l\'operation financiere executee (DEPOSIT, WITHDRAWAL, TRANSFER, PAYMENT).'
        },
        'transaction_amount': {
            'type': 'Numerique', 
            'description': 'Montant financier de l\'operation. Variable quantitative principale exploitee par les algorithmes pour mesurer l\'ecart statistique a la moyenne.'
        },
        'transaction_direction': {
            'type': 'Categoriel', 
            'description': 'Sens du flux financier du point de vue du compte du client. DEBIT = Sortie d\'argent, CREDIT = Entree d\'argent.'
        },
        'account_balance': {
            'type': 'Numerique', 
            'description': 'Solde restant sur le compte apres l\'execution de la transaction. Utile pour detecter les comportements de vidage rapide de compte.'
        },
        'merchant_category': {
            'type': 'Categoriel', 
            'description': 'Secteur d\'activite du commercant ou du beneficiaire de la transaction. Permet d\'identifier des habitudes d\'achat ou des commerces a haut risque.'
        },
        'state': {
            'type': 'Categoriel', 
            'description': 'Localisation geographique (Etat/Region) ou la transaction a ete initiee. Permet de detecter les incoherences geographiques.'
        },
        'credit_score': {
            'type': 'Numerique', 
            'description': 'Score de solvabilite ou cote de credit du client etablie par l\'institution. Generalement compris entre 300 et 850. Un score faible peut etre correle a un profil de risque different.'
        },
        'has_loan': {
            'type': 'Booleen', 
            'description': 'Indicateur signalant si le client possede un ou plusieurs emprunts actifs au sein de l\'etablissement. 0 = Pas de pret, 1 = Pret actif.'
        },
        'loan_type': {
            'type': 'Categoriel', 
            'description': 'Nature du pret contracte par le client si la variable has_loan est active (Home Loan, Personal Loan, Auto Loan, etc.).'
        },
        'emi_amount': {
            'type': 'Numerique', 
            'description': 'Equated Monthly Installment. Represente le montant mensuel fixe que le client doit rembourser pour ses emprunts.'
        },
        'transaction_status': {
            'type': 'Categoriel', 
            'description': 'Statut final de la transaction au niveau du systeme bancaire. SUCCESS = Validee, FAILED = Echouee, PENDING = En attente, REVERSED = Annulee.'
        },
        'channel': {
            'type': 'Categoriel', 
            'description': 'Canal ou interface utilise par le client pour initier l\'operation financiere. Mobile_App, Web, ATM, POS_Terminal, Branch, API.'
        },
        'kyc_status': {
            'type': 'Categoriel', 
            'description': 'Statut de conformite du dossier Know Your Customer. VERIFIED = Valide, PENDING = En attente, EXPIRED = Non conforme.'
        },
        'is_fraud': {
            'type': 'Booleen', 
            'description': 'Variable cible indiquant si la transaction est une fraude avere. 0 = Normal, 1 = Fraude. Strictement masquee durant l\'entrainement et reservee a la validation finale.'
        },
        'transaction_hour': {
            'type': 'Numerique', 
            'description': 'Variable numerique extrayant uniquement l\'heure (0 a 23) pour faciliter l\'analyse statistique des operations nocturnes.'
        }
    }
    
    data_dict = []
    for col in df.columns:
        col_info = {
            'Variable' : col,
            'Type_Technique' : str(df[col].dtype),
            'Type_Donnee' : variable_descriptions.get(col, {}).get('type', 'Inconnu'),
            'Description' : variable_descriptions.get(col, {}).get('description', 'A documenter'),
            'Valeurs_manquantes' : df[col].isnull().sum(),
            'Taux_manquants' : f"{df[col].isnull().sum() / len(df) * 100:.1f}%",
            'Valeurs_uniques' : df[col].nunique(),
            'Min' : df[col].min() if pd.api.types.is_numeric_dtype(df[col]) else 'N/A',
            'Max' : df[col].max() if pd.api.types.is_numeric_dtype(df[col]) else 'N/A'
        }
        data_dict.append(col_info)
    
    df_dict = pd.DataFrame(data_dict)
    
    # Affichage
    print("\nDICTIONNAIRE DES DONNEES :")
    display(df_dict)
    
    # Sauvegarde
    df_dict.to_csv('../data/processed/data_dictionary.csv', index=False, encoding='utf-8')
    print("\nDictionnaire sauvegarde : '../data/processed/data_dictionary.csv'")
    
    return df_dict

data_dict = create_data_dictionary(df)


5. DICTIONNAIRE DES VARIABLES

DICTIONNAIRE DES DONNEES :


,Variable,Type_Technique,Type_Donnee,Description,Valeurs_manquantes,Taux_manquants,Valeurs_uniques,Min,Max
0,transaction_id,object,Categoriel,Identifiant unique attribue a chaque transacti...,0,0.0%,550000,N/A,N/A
1,customer_id,object,Categoriel,Identifiant unique du client. Variable pivot q...,0,0.0%,79916,N/A,N/A
2,transaction_date,object,Temporel,Date calendaire a laquelle l'operation a ete e...,0,0.0%,1827,N/A,N/A
3,transaction_time,object,Temporel,"Heure precise (minutes, secondes) de l'executi...",0,0.0%,1440,N/A,N/A
4,account_type,object,Categoriel,Type de compte bancaire sur lequel l'operation...,0,0.0%,5,N/A,N/A
5,transaction_type,object,Categoriel,Nature commerciale ou technique de l'operation...,0,0.0%,10,N/A,N/A
6,transaction_amount,float64,Numerique,Montant financier de l'operation. Variable qua...,0,0.0%,374552,2.4,10000000.0
7,transaction_direction,object,Categoriel,Sens du flux financier du point de vue du comp...,0,0.0%,2,N/A,N/A
8,account_balance,float64,Numerique,Solde restant sur le compte apres l'execution ...,0,0.0%,536170,500.0,5000000.0
9,merchant_category,object,Categoriel,Secteur d'activite du commercant ou du benefic...,0,0.0%,15,N/A,N/A



Dictionnaire sauvegarde : '../data/processed/data_dictionary.csv'


### SECTION 6 : AUDIT DE QUALITE INITIAL

In [37]:
"""
    Cette section effectue un audit complet de la qualite des donnees :

        6.1. ANALYSE DE LA COMPLETUDE (Valeurs manquantes):
            - Identification des colonnes avec valeurs manquantes
            - Calcul du taux de valeurs manquantes
            - Evaluation de l'impact sur la robustesse du modele

        6.2. ANALYSE DE L'INTEGRITE (Doublons) :
            - Detection des lignes strictement identiques
            - Distinction entre anomalie technique et fraude par repetition
            - Decision de suppression ou conservation

        6.3. ANALYSE DES TYPES ET FORMATS :
            - Verification de la conformite des types
            - Detection des erreurs de codification
            - Identification des colonnes a convertir
"""

def quality_audit(df: pd.DataFrame) -> dict:
    """
    Audit complet de la qualite des donnees.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame a auditer
        
    Returns
    -------
    dict
        Dictionnaire contenant les resultats de l'audit :
        - missing_values : Liste des colonnes avec valeurs manquantes
        - duplicates : Nombre de lignes dupliquees
        - type_issues : Liste des problemes de types detectes
        
    Examples
    --------
    >>> audit_results = quality_audit(df)
    >>> print(audit_results['duplicates'])
    0
    
    Notes
    -----
    Cette fonction affiche egalement les resultats dans la console
    et fournit des recommandations basees sur les problemes detectes.
    """

    print("\n" + "="*60)
    print("6. AUDIT DE QUALITE INITIAL")
    print("="*60)
    
    audit_results = {}
    
    # 6.1. ANALYSE DE LA COMPLETUDE (Valeurs manquantes)
    print("\n6.1 ANALYSE DE LA COMPLETUDE (Valeurs manquantes) :")
    print("-"*40)
    
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({
        'Colonne': missing.index,
        'Manquantes': missing.values,
        'Pourcentage': missing_pct.values
    })
    missing_df = missing_df[missing_df['Manquantes'] > 0].sort_values('Manquantes', ascending=False)
    
    if len(missing_df) > 0:
        print("Colonnes avec valeurs manquantes:")
        display(missing_df)
        audit_results['missing_values'] = missing_df.to_dict('records')
        
        print("\nImpact sur la robustesse du modele:")
        print("   - Les valeurs manquantes peuvent biaiser les calculs de distance")
        print("   - Un taux > 5% necessite une attention particuliere")
        print("   - Decision: Imputation par la mediane pour les numeriques")
        print("   - Decision: Imputation par 'Unknown' pour les categoriels")
    else:
        print("Aucune valeur manquante detectee - Excellente completude")
        audit_results['missing_values'] = []
    
    # 6.2. ANALYSE DE L'INTEGRITE (Doublons)
    print("\n6.2 ANALYSE DE L'INTEGRITE (Doublons) :")
    print("-"*40)
    
    duplicates = df.duplicated().sum()
    duplicate_pct = duplicates / len(df) * 100
    
    print(f"   - Lignes dupliquees: {duplicates:,} ({duplicate_pct:.2f}%)")
    
    if duplicates > 0:
        print("   Des doublons detectes (distinction entre anomalie technique et fraude par repetition)")
        print("   - Les doublons peuvent indiquer:")
        print("     * Erreurs de saisie ou de traitement")
        print("     * Tentatives de fraude par repetition")
        print("     * Transactions identiques legitimes")
        print("   - Decision : Suppression des doublons pour eviter le biais")
        print("   - Justification : Les doublons peuvent fausser les analyses statistiques")
        print("   - Exception : Conserver si les timestamps sont differents")
        audit_results['duplicates'] = duplicates
    else:
        print("   Aucun doublon detecte - Integrite parfaite")
        audit_results['duplicates'] = 0
    
    # 6.3. ANALYSE DES TYPES ET FORMATS
    print("\n6.3 ANALYSE DES TYPES ET FORMATS :")
    print("-"*40)
    
    type_issues = []
    for col in df.columns:
        if df[col].dtype == 'object':
            # Verifier si la colonne peut etre convertie en date
            try:
                pd.to_datetime(df[col])
                print(f"   {col}: Peut etre converti en datetime")
            except:
                # Verifier les valeurs uniques suspectes
                unique_vals = df[col].unique()
                if len(unique_vals) > 50:
                    print(f"   {col} : {len(unique_vals)} valeurs uniques (texte libre)")
                else:
                    print(f"   {col} : {len(unique_vals)} valeurs uniques (categoriel)")
        elif pd.api.types.is_numeric_dtype(df[col]):
            # Verifier les valeurs negatives inattendues
            if (df[col] < 0).any():
                neg_count = (df[col] < 0).sum()
                print(f"   {col} : {neg_count} valeurs negatives")
                type_issues.append(f"{col}: valeurs negatives")
                print(f"   - Decision : Conserver les valeurs negatives")
                print(f"   - Justification : Les soldes negatifs peuvent etre legitimes (decouvert)")
            else:
                print(f"   {col}: Numerique valide")
    
    audit_results['type_issues'] = type_issues
    
    # 6.4. SYNTHESE DE L'AUDIT
    print("\nSYNTHESE DE L'AUDIT DE QUALITE :")
    print("-"*40)
    print(f"   Completude : {'Aucun probleme' if len(missing_df) == 0 else f'{len(missing_df)} colonnes avec valeurs manquantes'}")
    print(f"   Integrite : {'Aucun doublon' if duplicates == 0 else f'{duplicates:,} doublons detectes'}")
    print(f"   Types : {'Aucun probleme' if len(type_issues) == 0 else f'{len(type_issues)} problemes detectes'}")
    
    return audit_results

audit_results = quality_audit(df)


6. AUDIT DE QUALITE INITIAL

6.1 ANALYSE DE LA COMPLETUDE (Valeurs manquantes) :
----------------------------------------
Colonnes avec valeurs manquantes:


,Colonne,Manquantes,Pourcentage
13,loan_type,377134,68.569818



Impact sur la robustesse du modele:
   - Les valeurs manquantes peuvent biaiser les calculs de distance
   - Un taux > 5% necessite une attention particuliere
   - Decision: Imputation par la mediane pour les numeriques
   - Decision: Imputation par 'Unknown' pour les categoriels

6.2 ANALYSE DE L'INTEGRITE (Doublons) :
----------------------------------------
   - Lignes dupliquees: 0 (0.00%)
   Aucun doublon detecte - Integrite parfaite

6.3 ANALYSE DES TYPES ET FORMATS :
----------------------------------------
   transaction_id : 550000 valeurs uniques (texte libre)
   customer_id : 79916 valeurs uniques (texte libre)
   transaction_date: Peut etre converti en datetime
   transaction_time: Peut etre converti en datetime
   account_type : 5 valeurs uniques (categoriel)
   transaction_type : 10 valeurs uniques (categoriel)
   transaction_amount: Numerique valide
   transaction_direction : 2 valeurs uniques (categoriel)
   account_balance: Numerique valide
   merchant_category : 15 v

### SECTION 7: PROTOCOLE ET REGLES DE NETTOYAGE

In [38]:
"""

    Cette section definit et justifie les regles de nettoyage :

        7.1. PRINCIPE DE NON-ALTERATION DE LA SOURCE BRUTE:
            - Creation d'une version propre sans ecraser les donnees brutes
            - Fichier source: ../data/raw/indian_banking_transactions.csv
            - Fichier destination: ../data/processed/indian_banking_transactions_clean.csv

        7.2. REGLES DE NETTOYAGE:
            - Doublons: Suppression des lignes strictement identiques
            - Valeurs manquantes: Imputation selon le type de variable
            - Valeurs extremes: MAINTIEN pour l'apprentissage non supervise
            - Types: Correction des types (datetime, category)

        7.3. JUSTIFICATION:
            - Chaque regle est justifiee par des considerations statistiques
            - Impact sur l'analyse est documente
            - Les decisions sont tracables
"""

def define_cleaning_rules(df: pd.DataFrame) -> dict:
    """
    Definit et justifie les regles de nettoyage.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame source (non utilise directement mais garde pour reference)
        
    Returns
    -------
    dict
        Dictionnaire contenant les regles de nettoyage:
        - principe_non_alteration: Regle de non-alteration de la source
        - doublons: Regle de gestion des doublons
        - valeurs_manquantes: Regle d'imputation
        - valeurs_extremes_outliers: Regle de conservation
        - types_formats: Regle de correction des types
        
    Examples
    --------
    >>> rules = define_cleaning_rules(df)
    >>> print(rules['doublons']['regle'])
    Suppression des lignes strictement identiques
    
    Notes
    -----
    Les regles sont egalement sauvegardees dans un fichier texte.
    """
    print("\n" + "="*60)
    print("7. PROTOCOLE ET REGLES DE NETTOYAGE")
    print("="*60)
    
    rules = {
        'principe_non_alteration' : {
            'regle' : 'Creation d\'une version propre sans ecraser les donnees brutes',
            'justification' : 'Conformite avec les bonnes pratiques d\'ingenierie des donnees',
            'fichier_source' : '../data/raw/indian_banking_transactions.csv',
            'fichier_destination' : '../data/processed/indian_banking_transactions_clean.csv'
        },
        'doublons': {
            'regle' : 'Suppression des lignes strictement identiques',
            'justification' : 'Les doublons peuvent fausser les analyses et les modeles',
            'exceptions' : 'Conserver les transactions identiques avec des timestamps differents',
            'impact' : 'Reduction du nombre de lignes, amelioration de la qualite'
        },
        'valeurs_manquantes': {
            'regle' : 'Traitement specifique selon le type de variable',
            'justification' : 'Preserver l\'integrite des donnees et eviter les biais',
            'pour_categorielles' : 'Remplacer par "Unknown" (maintien de la categorie)',
            'pour_numeriques' : 'Remplacer par la mediane (robuste aux valeurs aberrantes)',
            'pour_temporelles' : 'Conserver ou imputer par la valeur la plus frequente',
            'impact' : 'Maintien de la taille du dataset, reduction des biais'
        },
        'valeurs_extremes_outliers': {
            'regle' : 'MAINTIEN DES MONTANTS GEANTS',
            'justification' : 'Les anomalies financieres recherchees par l\'apprentissage non supervise',
            'approche' : 'Detection sans suppression (conservation pour Isolation Forest)',
            'traitement' : 'Bornage (clipping) pour les valeurs extremes uniquement si necessaire',
            'impact' : 'Conservation des anomalies potentielles pour la modelisation'
        },
        'types_formats': {
            'regle' : 'Correction des types de donnees',
            'justification' : 'Assurer la compatibilite avec les algorithmes',
            'actions' : [
                'Convertir les dates en datetime',
                'Convertir les categorielles en type category',
                'Normaliser les formats de texte'
            ],
            'impact' : 'Amelioration de la performance des modeles'
        }
    }
    
    print("\nREGLES DE NETTOYAGE :")
    for key, rule in rules.items() :
        print(f"\n{key.upper()} :")
        for sub_key, value in rule.items():
            print(f"   - {sub_key} : {value}")
    
    # Sauvegarde des regles
    with open('../data/processed/cleaning_rules.txt', 'w', encoding='utf-8') as f:
        f.write("PROTOCOLE ET REGLES DE NETTOYAGE\n")
        f.write("="*50 + "\n\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        for key, rule in rules.items():
            f.write(f"{key.upper()}:\n")
            for sub_key, value in rule.items():
                f.write(f"  - {sub_key}: {value}\n")
            f.write("\n")
    
    print("\nRegles sauvegardees : '../data/processed/cleaning_rules.txt'")
    
    return rules

cleaning_rules = define_cleaning_rules(df)


7. PROTOCOLE ET REGLES DE NETTOYAGE

REGLES DE NETTOYAGE :

PRINCIPE_NON_ALTERATION :
   - regle : Creation d'une version propre sans ecraser les donnees brutes
   - justification : Conformite avec les bonnes pratiques d'ingenierie des donnees
   - fichier_source : ../data/raw/indian_banking_transactions.csv
   - fichier_destination : ../data/processed/indian_banking_transactions_clean.csv

DOUBLONS :
   - regle : Suppression des lignes strictement identiques
   - justification : Les doublons peuvent fausser les analyses et les modeles
   - exceptions : Conserver les transactions identiques avec des timestamps differents
   - impact : Reduction du nombre de lignes, amelioration de la qualite

VALEURS_MANQUANTES :
   - regle : Traitement specifique selon le type de variable
   - justification : Preserver l'integrite des donnees et eviter les biais
   - pour_categorielles : Remplacer par "Unknown" (maintien de la categorie)
   - pour_numeriques : Remplacer par la mediane (robuste aux va

### SECTION 8: EXECUTION DU NETTOYAGE

In [ ]:
"""
    Cette section execute les regles de nettoyage definies :

        8.1. SUPPRESSION DES DOUBLONS :
            - Suppression des lignes strictement identiques
            - Tracabilite du nombre de lignes supprimees

        8.2. GESTION DES VALEURS MANQUANTES:
            - Imputation des valeurs manquantes
            - Journalisation des decisions

        8.3. CORRECTION DES TYPES :
            - Conversion des dates en datetime
            - Maintien des types appropries

        8.4. TRAITEMENT DES VALEURS EXTREMES :
            - Conservation des montants geants
            - Bornage (clipping) si necessaire

        8.5. SAUVEGARDE:
            - Fichier nettoye : ../data/processed/indian_banking_transactions_clean.csv
            - Journal des operations: ../data/processed/cleaning_log.txt
"""

def execute_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    """
    Execute le nettoyage selon les regles definies.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame brut a nettoyer
        
    Returns
    -------
    pd.DataFrame
        DataFrame nettoye
        
    Notes
    -----
    Cette fonction applique les regles de nettoyage definies dans la section 7.
    Toutes les operations sont journalisees pour la tracabilite.
    
    Examples
    --------
    >>> df_clean = execute_cleaning(df)
    >>> print(f"Lignes avant: {len(df)}, Lignes apres: {len(df_clean)}")
    
    Raises
    ------
    Exception
        Si une erreur survient pendant le nettoyage
    """

    print("\n" + "="*60)
    print("8. EXECUTION DU NETTOYAGE")
    print("="*60)
    
    # Copie pour le nettoyage
    df_clean = df.copy()
    
    # Journal des operations
    cleaning_log = []
    
    # 8.1. Suppression des doublons
    print("\n8.1 SUPPRESSION DES DOUBLONS :")
    duplicates_before = df_clean.duplicated().sum()
    if duplicates_before > 0:
        df_clean = df_clean.drop_duplicates()
        duplicates_after = df_clean.duplicated().sum()
        removed = duplicates_before - duplicates_after
        print(f"   Doublons supprimes : {removed:,}")
        cleaning_log.append(f"Doublons supprimes : {removed} lignes")
    else:
        print("   Aucun doublon a supprimer")
    
    # 8.2. Gestion des valeurs manquantes
    print("\n8.2 GESTION DES VALEURS MANQUANTES :")
    missing_before = df_clean.isnull().sum().sum()
    
    if missing_before > 0:
        for col in df_clean.columns:
            if df_clean[col].isnull().any():
                if df_clean[col].dtype == 'object':
                    df_clean[col] = df_clean[col].fillna('Unknown')
                    print(f"   {col} : Remplace par 'Unknown' (categoriel)")
                    cleaning_log.append(f"{col}: 'Unknown' pour valeurs manquantes")
                else:
                    median_val = df_clean[col].median()
                    df_clean[col] = df_clean[col].fillna(median_val)
                    print(f"   {col} : Remplace par mediane ({median_val:.2f})")
                    cleaning_log.append(f"{col}: mediane ({median_val:.2f})")
    else:
        print("   Aucune valeur manquante a traiter")
    
    # 8.3. Correction des types
    print("\n8.3 CORRECTION DES TYPES :")
    for col in df_clean.columns:
        if df_clean[col].dtype == 'object':
            try:
                df_clean[col] = pd.to_datetime(df_clean[col])
                print(f"   {col} : Converti en datetime")
                cleaning_log.append(f"{col}: converti en datetime")
            except:
                print(f"   {col} : Maintien en format texte")
    
    # 8.4. Traitement des valeurs extremes (MAINTIEN)
    print("\n8.4 TRAITEMENT DES VALEURS EXTREMES :")
    print("   PRINCIPE: Maintien des montants geants pour l'apprentissage non supervise")
    print("   JUSTIFICATION: Les anomalies financieres recherchees sont les valeurs extremes")
    print("   IMPACT: Conservation des points aberrants pour Isolation Forest et DBSCAN")
    
    for col in df_clean.select_dtypes(include=[np.number]).columns:
        mean_val = df_clean[col].mean()
        std_val = df_clean[col].std()
        
        # Calcul des limites (3 sigma)
        lower_bound = mean_val - 3 * std_val
        upper_bound = mean_val + 3 * std_val
        
        outliers = ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()
        
        if outliers > 0:
            print(f"   {col} : {outliers} valeurs au-dela de 3sigma")
            print(f"      - Moyenne : {mean_val:.2f}, Ecart-type: {std_val:.2f}")
            print(f"      - Bornes : [{lower_bound:.2f}, {upper_bound:.2f}]")
            print(f"      - DECISION : Conservation (anomalies potentielles)")
            cleaning_log.append(f"{col} : {outliers} valeurs extremes conservees pour detection d'anomalies")
    
    # 8.5. Statistiques finales
    print("\nSTATISTIQUES FINALES:")
    print("-"*40)
    print(f"   - Lignes avant nettoyage : {len(df):,}")
    print(f"   - Lignes apres nettoyage : {len(df_clean):,}")
    print(f"   - Lignes supprimees : {len(df) - len(df_clean):,}")
    
    # Sauvegarde du journal
    with open('../data/processed/cleaning_log.txt', 'w', encoding='utf-8') as f:
        f.write("JOURNAL DE NETTOYAGE\n")
        f.write("="*50 + "\n")
        f.write(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write("OPERATIONS EFFECTUEES :\n")
        f.write("-"*40 + "\n")
        for log_entry in cleaning_log:
            f.write(f"  {log_entry}\n")
        f.write(f"\nTotal : {len(cleaning_log)} operations effectuees")
    
    print("\nJournal sauvegarde : '../data/processed/cleaning_log.txt'")
    
    return df_clean

# Execution du nettoyage
df_clean = execute_cleaning(df)

# Sauvegarde des donnees nettoyees
df_clean.to_csv('../data/processed/indian_banking_transactions_clean.csv', index=False)
print("\nDonnees nettoyees sauvegardees: '../data/processed/indian_banking_transactions_clean.csv'")



8. EXECUTION DU NETTOYAGE

8.1 SUPPRESSION DES DOUBLONS :
   Aucun doublon a supprimer

8.2 GESTION DES VALEURS MANQUANTES :
   loan_type : Remplace par 'Unknown' (categoriel)

8.3 CORRECTION DES TYPES :
   transaction_id : Maintien en format texte
   customer_id : Maintien en format texte
   transaction_date : Converti en datetime


### SECTION 9 : STATISTIQUES DESCRIPTIVES

In [ ]:
"""
    Cette section produit des statistiques descriptives:

        9.1. VARIABLES NUMERIQUES:
            - Moyenne, mediane, ecart-type
            - Minimum, maximum, quartiles
            - Asymetrie (skewness)
            - Aplatissement (kurtosis)
            - Interpretation des distributions

        9.2. VARIABLES CATEGORIELLES:
            - Frequences des valeurs
            - Distribution des categories
            - Identification des valeurs dominantes
"""

def descriptive_analysis(df: pd.DataFrame) -> None:
    """
    Analyse descriptive complete des donnees.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame a analyser
        
    Returns
    -------
    None
        Les resultats sont affiches et sauvegardes.
        
    Notes
    -----
    Cette fonction produit:
        - Statistiques descriptives pour les variables numeriques
        - Frequences pour les variables categorielles
        - Sauvegarde des resultats dans results/metrics/
    
    Examples
    --------
    >>> descriptive_analysis(df_clean)
    """
    print("\n" + "="*60)
    print("9. STATISTIQUES DESCRIPTIVES")
    print("="*60)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 0:
        print("\n9.1 STATISTIQUES DESCRIPTIVES - VARIABLES NUMERIQUES:")
        print("-"*40)
        print("Les statistiques suivantes permettent de comprendre la tendance centrale")
        print("et la dispersion des variables numeriques.")
        display(df[numeric_cols].describe().round(2))
        
        # Sauvegarde
        df[numeric_cols].describe().to_csv('../results/metrics/descriptive_stats_numeric.csv')
        print("\nStatistiques sauvegardees: '../results/metrics/descriptive_stats_numeric.csv'")
    
    # Variables categorielles
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    
    if len(cat_cols) > 0:
        print("\n9.2 STATISTIQUES DESCRIPTIVES - VARIABLES CATEGORIELLES:")
        print("-"*40)
        print("Les frequences des categories permettent d'identifier les valeurs dominantes.")
        for col in cat_cols[:5]:
            print(f"\n{col}:")
            display(df[col].value_counts().head(10))
            
            # Sauvegarde
            df[col].value_counts().to_csv(f'../results/metrics/{col}_distribution.csv')
            print(f"   Sauvegarde: '../results/metrics/{col}_distribution.csv'")

descriptive_analysis(df_clean)


9. STATISTIQUES DESCRIPTIVES

9.1 STATISTIQUES DESCRIPTIVES - VARIABLES NUMERIQUES:
----------------------------------------
Les statistiques suivantes permettent de comprendre la tendance centrale
et la dispersion des variables numeriques.


,transaction_amount,account_balance,credit_score,has_loan,emi_amount,is_fraud,transaction_hour
count,550000.00,550000.00,550000.00,550000.00,550000.00,550000.00,550000.00
mean,29907.09,84550.12,599.81,0.35,2365.86,0.01,11.50
std,139610.10,170914.59,173.05,0.48,4973.55,0.09,6.92
min,2.40,500.00,300.00,0.00,0.00,0.00,0.00
25%,763.04,15157.78,450.00,0.00,0.00,0.00,6.00
50%,2035.74,36422.87,600.00,0.00,0.00,0.00,11.00
75%,8557.50,87607.04,750.00,1.00,3115.03,0.00,17.00
max,10000000.00,5000000.00,899.00,1.00,153325.74,1.00,23.00



Statistiques sauvegardees: '../results/metrics/descriptive_stats_numeric.csv'

9.2 STATISTIQUES DESCRIPTIVES - VARIABLES CATEGORIELLES:
----------------------------------------
Les frequences des categories permettent d'identifier les valeurs dominantes.

transaction_id:


transaction_id
TXN000550000    1
TXN000000001    1
TXN000000002    1
TXN000000003    1
TXN000000004    1
TXN000000005    1
TXN000000006    1
TXN000000007    1
TXN000000008    1
TXN000000009    1
Name: count, dtype: int64

   Sauvegarde: '../results/metrics/transaction_id_distribution.csv'

customer_id:


...

   Sauvegarde: '../results/metrics/account_type_distribution.csv'

transaction_type:


transaction_type
UPI               153986
IMPS               77038
NEFT               66615
POS                60278
ATM_Withdrawal     55106
Net_Banking        38458
RTGS               32964
Auto_Debit         27418
Cheque             21711
Credit_Card        16426
Name: count, dtype: int64

   Sauvegarde: '../results/metrics/transaction_type_distribution.csv'

transaction_direction:


transaction_direction
Debit     370394
Credit    179606
Name: count, dtype: int64

   Sauvegarde: '../results/metrics/transaction_direction_distribution.csv'


### SECTION 10 : ANALYSE DES DISTRIBUTIONS

In [ ]:
"""
    Cette section analyse les distributions des variables:

        10.1. HISTOGRAMMES:
            - Visualisation de la distribution de chaque variable numerique
            - Identification de la moyenne et de la mediane
            - Detection des asymetries

        10.2. BOITES A MOUSTACHES (BOXPLOTS):
            - Identification des valeurs aberrantes
            - Visualisation des quartiles
            - Detection des anomalies potentielles
"""

def analyze_distributions(df: pd.DataFrame) -> None:
    """
    Analyse et visualise les distributions des variables.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame a analyser
        
    Returns
    -------
    None
        Les graphiques sont affiches et sauvegardes.
        
    Notes
    -----
    Cette fonction produit:
        - Histogrammes pour les variables numeriques
        - Boxplots pour les variables numeriques
        - Sauvegarde dans results/figures/
    
    Examples
    --------
    >>> analyze_distributions(df_clean)
    """
    print("\n" + "="*60)
    print("10. ANALYSE DES DISTRIBUTIONS")
    print("="*60)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) == 0:
        print("Aucune variable numerique")
        return
    
    print("\n10.1 HISTOGRAMMES DES DISTRIBUTIONS:")
    print("-"*40)
    print("Les histogrammes permettent de visualiser la forme de la distribution")
    print("et d'identifier les asymetries ou les valeurs extremes.")
    
    # Histogrammes
    n_cols = min(len(numeric_cols), 4)
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    if n_rows == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols[:len(axes)]):
        ax = axes[i]
        ax.hist(df[col].dropna(), bins=50, alpha=0.7, color='#2E86AB', edgecolor='black', linewidth=0.5)
        ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=2, label=f'Moyenne: {df[col].mean():.2f}')
        ax.axvline(df[col].median(), color='green', linestyle='--', linewidth=2, label=f'Mediane: {df[col].median():.2f}')
        ax.set_title(f'Distribution de {col}', fontsize=12)
        ax.set_xlabel(col)
        ax.set_ylabel('Frequence')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('../results/figures/distributions.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("\nGraphique sauvegarde: '../results/figures/distributions.png'")
    
    # Boxplots
    print("\n10.2 BOITES A MOUSTACHES (BOXPLOTS):")
    print("-"*40)
    print("Les boxplots permettent d'identifier les valeurs aberrantes (outliers)")
    print("et de visualiser la dispersion des donnees.")
    
    n_cols = min(len(numeric_cols), 3)
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    if n_rows == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols[:len(axes)]):
        ax = axes[i]
        ax.boxplot(df[col].dropna(), patch_artist=True,
                   boxprops=dict(facecolor='#2E86AB', alpha=0.7))
        ax.set_title(f'Boite a moustaches de {col}', fontsize=12)
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3)
    
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('../results/figures/boxplots.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Graphique sauvegarde: '../results/figures/boxplots.png'")

analyze_distributions(df_clean)


10. ANALYSE DES DISTRIBUTIONS

10.1 HISTOGRAMMES DES DISTRIBUTIONS:
----------------------------------------
Les histogrammes permettent de visualiser la forme de la distribution
et d'identifier les asymetries ou les valeurs extremes.

Graphique sauvegarde: '../results/figures/distributions.png'

10.2 BOITES A MOUSTACHES (BOXPLOTS):
----------------------------------------
Les boxplots permettent d'identifier les valeurs aberrantes (outliers)
et de visualiser la dispersion des donnees.
Graphique sauvegarde: '../results/figures/boxplots.png'


### SECTION 11 : ANALYSE DES RELATIONS ENTRE VARIABLES

In [ ]:
"""
    Cette section analyse les relations entre les variables:

        11.1. MATRICE DE CORRELATION:
            - Calcul des correlations entre variables numeriques
            - Identification des relations lineaires

        11.2. CARTE DE CHALEUR (HEATMAP):
            - Visualisation de la matrice de correlation
            - Identification des correlations fortes

        11.3. CORRELATIONS FORTES:
            - Detection des |r| > 0.7
            - Interpretation des relations
"""

def analyze_relationships(df: pd.DataFrame) -> None:
    """
    Analyse les relations entre les variables.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame a analyser
        
    Returns
    -------
    None
        Les resultats sont affiches et sauvegardes.
        
    Notes
    -----
    Cette fonction produit:
        - Matrice de correlation
        - Heatmap des correlations
        - Sauvegarde dans results/figures/ et results/metrics/
    
    Examples
    --------
    >>> analyze_relationships(df_clean)
    """
    print("\n" + "="*60)
    print("11. ANALYSE DES RELATIONS ENTRE VARIABLES")
    print("="*60)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) < 2:
        print("Pas assez de variables numeriques")
        return
    
    # 11.1. Matrice de correlation
    print("\n11.1 MATRICE DE CORRELATION:")
    print("-"*40)
    print("La correlation mesure la relation lineaire entre deux variables.")
    print("  - r > 0: Relation positive")
    print("  - r < 0: Relation negative")
    print("  - r = 0: Pas de relation lineaire")
    
    corr_matrix = df[numeric_cols].corr()
    display(corr_matrix.round(2))
    corr_matrix.to_csv('../results/metrics/correlation_matrix.csv')
    
    # 11.2. Heatmap
    print("\n11.2 HEATMAP) :")
    print("-"*40)
    print("La heatmap permet de visualiser rapidement les correlations.")
    
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
                cmap='RdBu_r', center=0, square=True,
                linewidths=0.5)
    plt.title('Matrice de Correlation des Variables Numeriques', fontsize=16)
    plt.tight_layout()
    plt.savefig('../results/figures/correlation_heatmap.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("\nGraphique sauvegarde: '../results/figures/correlation_heatmap.png'")
    
    # 11.3. Correlations fortes
    print("\n11.3 CORRELATIONS FORTES (|r| > 0.7):")
    print("-"*40)
    strong_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.7:
                strong_corr.append({
                    'Variable1': corr_matrix.columns[i],
                    'Variable2': corr_matrix.columns[j],
                    'Correlation': corr_matrix.iloc[i, j]
                })
    
    if strong_corr:
        print("Des correlations fortes detectees:")
        for item in strong_corr:
            print(f"   - {item['Variable1']} <-> {item['Variable2']}: {item['Correlation']:.3f}")
        print("   -> Ces variables sont fortement liees et pourraient etre redondantes.")
    else:
        print("Aucune correlation forte detectee")
        print("   -> Les variables sont globalement independantes.")
    
    # 11.4. Interpretation
    print("\n11.4 INTERPRETATION:")
    print("-"*40)
    print("   - Les correlations moderees indiquent des relations lineaires interessantes")
    print("   - L'absence de correlations fortes suggere une redondance limitee")
    print("   - Les variables sont globalement independantes, ce qui est favorable")
    print("   - Les algorithmes de detection d'anomalies beneficieront de cette diversite")

analyze_relationships(df_clean)


11. ANALYSE DES RELATIONS ENTRE VARIABLES

11.1 MATRICE DE CORRELATION:
----------------------------------------
La correlation mesure la relation lineaire entre deux variables.
  - r > 0: Relation positive
  - r < 0: Relation negative
  - r = 0: Pas de relation lineaire


,transaction_amount,account_balance,credit_score,has_loan,emi_amount,is_fraud,transaction_hour
transaction_amount,1.00,-0.0,0.0,0.00,-0.00,0.06,-0.0
account_balance,-0.00,1.0,0.0,-0.00,-0.00,0.00,0.0
credit_score,0.00,0.0,1.0,0.00,0.00,0.00,0.0
has_loan,0.00,-0.0,0.0,1.00,0.65,0.00,0.0
emi_amount,-0.00,-0.0,0.0,0.65,1.00,0.00,0.0
is_fraud,0.06,0.0,0.0,0.00,0.00,1.00,-0.0
transaction_hour,-0.00,0.0,0.0,0.00,0.00,-0.00,1.0



11.2 HEATMAP) :
----------------------------------------
La heatmap permet de visualiser rapidement les correlations.

Graphique sauvegarde: '../results/figures/correlation_heatmap.png'

11.3 CORRELATIONS FORTES (|r| > 0.7):
----------------------------------------
Aucune correlation forte detectee
   -> Les variables sont globalement independantes.

11.4 INTERPRETATION:
----------------------------------------
   - Les correlations moderees indiquent des relations lineaires interessantes
   - L'absence de correlations fortes suggere une redondance limitee
   - Les variables sont globalement independantes, ce qui est favorable
   - Les algorithmes de detection d'anomalies beneficieront de cette diversite


### SECTION 12 : DETECTION PRELIMINAIRE DES ANOMALIES

In [ ]:
"""
    Cette section effectue une detection preliminaire des anomalies:

        12.1. NUAGES DE POINTS (SCATTER PLOTS):
            - Visualisation des relations entre paires de variables
            - Identification des points aberrants
            - Detection des clusters et des valeurs extremes

        12.2. VALEURS EXTREMES:
            - Identification des valeurs au-dela de 3sigma
            - Comptage des valeurs extremes
            - Conservation pour les modeles non supervises
"""

def detect_anomalies_preliminary(df: pd.DataFrame) -> None:
    """
    Detection visuelle des anomalies potentielles.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame a analyser
        
    Returns
    -------
    None
        Les graphiques sont affiches et sauvegardes.
        
    Notes
    -----
    Cette fonction produit:
        - Scatter plots pour les paires de variables
        - Identification des valeurs extremes
        - Sauvegarde dans results/figures/
    
    Examples
    --------
    >>> detect_anomalies_preliminary(df_clean)
    """
    print("\n" + "="*60)
    print("12. DETECTION PRELIMINAIRE DES ANOMALIES")
    print("="*60)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) < 2:
        print("Pas assez de variables numeriques")
        return
    
    # 12.1. Nuages de points (Scatter plots)
    print("\n12.1 NUAGES DE POINTS (SCATTER PLOTS):")
    print("-"*40)
    print("Les scatter plots permettent de visualiser les relations entre paires de variables.")
    print("Les points isoles peuvent representer des anomalies potentielles.")
    
    n_pairs = min(len(numeric_cols) - 1, 6)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    plot_idx = 0
    for i in range(len(numeric_cols) - 1):
        if plot_idx >= n_pairs:
            break
        for j in range(i + 1, len(numeric_cols)):
            if plot_idx >= n_pairs:
                break
            
            ax = axes[plot_idx]
            x_col, y_col = numeric_cols[i], numeric_cols[j]
            
            ax.scatter(df[x_col], df[y_col], alpha=0.5, s=10)
            ax.set_xlabel(x_col)
            ax.set_ylabel(y_col)
            ax.set_title(f'{x_col} vs {y_col}')
            ax.grid(True, alpha=0.3)
            
            plot_idx += 1
    
    for idx in range(plot_idx, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('../results/figures/anomalies_scatter.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("\nGraphique sauvegarde: '../results/figures/anomalies_scatter.png'")
    
    # 12.2. Valeurs extremes
    print("\n12.2 VALEURS EXTREMES POTENTIELLES (au-dela de 3sigma):")
    print("-"*40)
    print("Les valeurs au-dela de 3sigma sont considerees comme des anomalies potentielles.")
    print("Elles seront conservees pour les modeles non supervises.")
    
    for col in numeric_cols:
        mean = df[col].mean()
        std = df[col].std()
        
        extreme_high = df[df[col] > mean + 3*std]
        extreme_low = df[df[col] < mean - 3*std]
        
        if len(extreme_high) > 0:
            print(f"   {col}: {len(extreme_high)} valeurs > {mean + 3*std:.2f} (3sigma)")
        if len(extreme_low) > 0:
            print(f"   {col}: {len(extreme_low)} valeurs < {mean - 3*std:.2f} (3sigma)")
    
    # 12.3. Interpretation
    print("\n12.3 INTERPRETATION:")
    print("-"*40)
    print("   - Les valeurs extremes identifiees representent des anomalies potentielles")
    print("   - Ces points seront analyses par Isolation Forest et DBSCAN")
    print("   - La conservation de ces valeurs est essentielle pour la detection d'anomalies")
    print("   - Les modeles non supervises sont concus pour identifier ces points aberrants")

detect_anomalies_preliminary(df_clean)



12. DETECTION PRELIMINAIRE DES ANOMALIES

12.1 NUAGES DE POINTS (SCATTER PLOTS):
----------------------------------------
Les scatter plots permettent de visualiser les relations entre paires de variables.
Les points isoles peuvent representer des anomalies potentielles.

Graphique sauvegarde: '../results/figures/anomalies_scatter.png'

12.2 VALEURS EXTREMES POTENTIELLES (au-dela de 3sigma):
----------------------------------------
Les valeurs au-dela de 3sigma sont considerees comme des anomalies potentielles.
Elles seront conservees pour les modeles non supervises.
   transaction_amount: 7143 valeurs > 448737.38 (3sigma)
   account_balance: 8553 valeurs > 597293.87 (3sigma)
   emi_amount: 11178 valeurs > 17286.50 (3sigma)
   is_fraud: 4873 valeurs > 0.29 (3sigma)

12.3 INTERPRETATION:
----------------------------------------
   - Les valeurs extremes identifiees representent des anomalies potentielles
   - Ces points seront analyses par Isolation Forest et DBSCAN
   - La conservatio

### SECTION 13 : SYNTHESE DES RESULTATS

In [ ]:
"""
    Cette section resume les resultats de l'analyse exploratoire:

        13.1. RESUME DES DONNEES:
            - Lignes originales vs nettoyees
            - Variables numeriques et categorielles

        13.2. PRINCIPAUX CONSTATS:
            - Qualite des donnees
            - Distributions
            - Relations entre variables
            - Anomalies potentielles

        13.3. RECOMMANDATIONS:
            - Modeles a utiliser
            - Pretraitements necessaires
            - Prochaines etapes
"""

def generate_summary(df_original: pd.DataFrame, df_clean: pd.DataFrame) -> None:
    """
    Genere une synthese des resultats de l'analyse exploratoire.
    
    Parameters
    ----------
    df_original : pd.DataFrame
        DataFrame original (brut)
    df_clean : pd.DataFrame
        DataFrame nettoye
        
    Returns
    -------
    None
        Les resultats sont affiches et sauvegardes.
        
    Notes
    -----
    Cette fonction produit:
        - Synthese des donnees
        - Principaux constats
        - Recommandations
        - Sauvegarde dans results/metrics/summary_report.txt
    
    Examples
    --------
    >>> generate_summary(df, df_clean)
    """
    print("\n" + "="*60)
    print("13. SYNTHESE DES RESULTATS")
    print("="*60)
    
    print("\n13.1 RESUME DES DONNEES:")
    print("-"*40)
    print(f"   - Lignes originales: {df_original.shape[0]:,}")
    print(f"   - Lignes nettoyees: {df_clean.shape[0]:,}")
    print(f"   - Lignes supprimees: {df_original.shape[0] - df_clean.shape[0]:,}")
    
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    cat_cols = df_clean.select_dtypes(include=['object', 'category']).columns
    
    print(f"\n   - Variables numeriques: {len(numeric_cols)}")
    print(f"   - Variables categorielles: {len(cat_cols)}")
    
    print("\n13.2 PRINCIPAUX CONSTATS:")
    print("-"*40)
    print("   1. QUALITE DES DONNEES:")
    print("      - Aucune valeur manquante significative detectee")
    print("      - Doublons supprimes (si presents)")
    print("      - Types de donnees correctement formates")
    
    print("\n   2. DISTRIBUTIONS:")
    print("      - Distributions majoritairement asymetriques")
    print("      - Presence de valeurs extremes conservees")
    print("      - Variabilite significative dans les montants")
    
    print("\n   3. RELATIONS:")
    print("      - Correlations moderees entre certaines variables")
    print("      - Absence de redondance forte")
    print("      - Variables globalement independantes")
    
    print("\n   4. ANOMALIES POTENTIELLES:")
    print("      - Valeurs extremes identifiees au-dela de 3sigma")
    print("      - Points aberrants visibles dans les scatter plots")
    print("      - Conservation pour les modeles non supervises")
    
    print("\n13.3:")
    print("-"*40)
    print("   1. MODELES:")
    print("      - Utiliser Isolation Forest pour la detection d'anomalies")
    print("      - Utiliser DBSCAN pour la segmentation")
    print("      - Utiliser ACP pour la visualisation")
    
    print("\n   2. PRETRAITEMENTS:")
    print("      - Normaliser les donnees (StandardScaler)")
    print("      - Encoder les variables categorielles")
    print("      - Creer des variables de frequence par client")
    
    print("\n   3. PROCHAINES ETAPES:")
    print("      - Notebook 02: Isolation Forest")
    print("      - Notebook 03: DBSCAN + ACP")
    print("      - Notebook 04: Evaluation finale")
    
    # Sauvegarde
    with open('../results/metrics/summary_report.txt', 'w', encoding='utf-8') as f:
        f.write("="*60 + "\n")
        f.write("SYNTHESE DE L'ANALYSE EXPLORATOIRE\n")
        f.write("="*60 + "\n\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write("13.1 RESUME DES DONNEES:\n")
        f.write("-"*40 + "\n")
        f.write(f"Lignes originales: {df_original.shape[0]:,}\n")
        f.write(f"Lignes nettoyees: {df_clean.shape[0]:,}\n")
        f.write(f"Lignes supprimees: {df_original.shape[0] - df_clean.shape[0]:,}\n\n")
        f.write(f"Variables numeriques: {len(numeric_cols)}\n")
        f.write(f"Variables categorielles: {len(cat_cols)}\n\n")
        
        f.write("13.2 PRINCIPAUX CONSTATS:\n")
        f.write("-"*40 + "\n")
        f.write("1. QUALITE DES DONNEES:\n")
        f.write("   - Aucune valeur manquante significative\n")
        f.write("   - Doublons supprimes\n")
        f.write("   - Types correctement formates\n\n")
        
        f.write("2. DISTRIBUTIONS:\n")
        f.write("   - Distributions asymetriques\n")
        f.write("   - Valeurs extremes conservees\n")
        f.write("   - Variabilite significative\n\n")
        
        f.write("3. RELATIONS:\n")
        f.write("   - Correlations moderees\n")
        f.write("   - Absence de redondance\n")
        f.write("   - Variables independantes\n\n")
        
        f.write("4. ANOMALIES:\n")
        f.write("   - Valeurs au-dela de 3sigma identifiees\n")
        f.write("   - Points aberrants conserves\n\n")
        
        f.write("13.3 :\n")
        f.write("-"*40 + "\n")
        f.write("1. MODELES:\n")
        f.write("   - Isolation Forest (detection)\n")
        f.write("   - DBSCAN (segmentation)\n")
        f.write("   - ACP (visualisation)\n\n")
        
        f.write("2. PRETRAITEMENTS:\n")
        f.write("   - StandardScaler (normalisation)\n")
        f.write("   - Encodage des categorielles\n")
        f.write("   - Variables de frequence\n\n")
    
    print("\nSynthese sauvegardee: '../results/metrics/summary_report.txt'")

generate_summary(df, df_clean)



13. SYNTHESE DES RESULTATS

13.1 RESUME DES DONNEES:
----------------------------------------
   - Lignes originales: 550,000
   - Lignes nettoyees: 550,000
   - Lignes supprimees: 0

   - Variables numeriques: 7
   - Variables categorielles: 11

13.2 PRINCIPAUX CONSTATS:
----------------------------------------
   1. QUALITE DES DONNEES:
      - Aucune valeur manquante significative detectee
      - Doublons supprimes (si presents)
      - Types de donnees correctement formates

   2. DISTRIBUTIONS:
      - Distributions majoritairement asymetriques
      - Presence de valeurs extremes conservees
      - Variabilite significative dans les montants

   3. RELATIONS:
      - Correlations moderees entre certaines variables
      - Absence de redondance forte
      - Variables globalement independantes

   4. ANOMALIES POTENTIELLES:
      - Valeurs extremes identifiees au-dela de 3sigma
      - Points aberrants visibles dans les scatter plots
      - Conservation pour les modeles non sup

### CONCLUSION

---

    RESULTATS PRODUITS :

    1. DONNEES NETTOYEES :
            - ../data/processed/indian_banking_transactions_clean.csv
            - ../data/processed/data_dictionary.csv
            - ../data/processed/cleaning_rules.txt
            - ../data/processed/cleaning_log.txt

    2. VISUALISATIONS :
            - ../results/figures/distributions.png
            - ../results/figures/boxplots.png
            - ../results/figures/correlation_heatmap.png
            - ../results/figures/anomalies_scatter.png

    3. METRIQUES :
            - ../results/metrics/descriptive_stats_numeric.csv
            - ../results/metrics/correlation_matrix.csv
            - ../results/metrics/summary_report.txt

    PROCHAINES ETAPES :

            1. Notebook 02: Isolation Forest
                    - Entrainement du modele
                    - Detection des anomalies
                    - Visualisation des resultats

            2. Notebook 03 : DBSCAN + ACP
                    - Entrainement de DBSCAN
                    - Analyse ACP
                    - Visualisation des clusters

            3. Notebook 04 : Evaluation finale
                    - Comparaison des modeles
                    - Metriques de performance
                    - Rapport final

---